In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("titanic.csv")

In [5]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [6]:
df.drop(columns=['PassengerId' ,  'Name' , 'Ticket' , 'Cabin'] , inplace=True)

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  418 non-null    int64  
 1   Pclass    418 non-null    int64  
 2   Sex       418 non-null    object 
 3   Age       418 non-null    float64
 4   SibSp     418 non-null    int64  
 5   Parch     418 non-null    int64  
 6   Fare      418 non-null    float64
 7   Embarked  418 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 26.3+ KB


In [25]:
df['Fare'] = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))
# Filling the single missing Fare with the median of its Pclass

In [27]:
df['Age'] = df['Age'].fillna(df.groupby(['Pclass', 'Sex'])['Age'].transform('median'))
# Why: Generally, 1st-class passengers were older than 3rd-class passengers, and gender distributions often varied by age in that era.

In [20]:
df['Sex'].dtypes
df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='object')

In [21]:
for col in df.columns:
    print ( "type of coulmn " , col , "---" , df[col].dtypes)

type of coulmn  Survived --- int64
type of coulmn  Pclass --- int64
type of coulmn  Sex --- object
type of coulmn  Age --- float64
type of coulmn  SibSp --- int64
type of coulmn  Parch --- int64
type of coulmn  Fare --- float64
type of coulmn  Embarked --- object


In [24]:
for col in df.columns:
    if df[col].dtypes == "object" :
        print ( "valuecount of coulmn " , col , "---" , df[col].value_counts(),sum)

valuecount of coulmn  Sex --- Sex
male      266
female    152
Name: count, dtype: int64 <built-in function sum>
valuecount of coulmn  Embarked --- Embarked
S    270
C    102
Q     46
Name: count, dtype: int64 <built-in function sum>


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split



In [30]:


# Define feature variables and target
X = df.drop(columns=['Survived'])
y = df['Survived']

# Identify categorical and numerical columns
categorical_cols = ['Sex', 'Embarked']
numerical_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

# Create preprocessor for scaling and encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ]
)

# Create the pipeline with preprocessor and logistic regression model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])


In [31]:

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [32]:

# Train the pipeline
pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:

# Predict on test data
y_pred = pipeline.predict(X_test)


In [36]:
from sklearn.metrics import accuracy_score, confusion_matrix , classification_report

In [37]:
accuracy_score(y_test,y_pred)

1.0

In [38]:
confusion_matrix(y_test,y_pred)

array([[50,  0],
       [ 0, 34]])

In [39]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        34

    accuracy                           1.00        84
   macro avg       1.00      1.00      1.00        84
weighted avg       1.00      1.00      1.00        84



In [34]:

# Display the pipeline score
score = pipeline.score(X_test, y_test)
print(f"Accuracy: {score:.4f}")

Accuracy: 1.0000


In [40]:
from sklearn.neighbors import KNeighborsClassifier

In [41]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

In [42]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [43]:
# Predict on test data
y_pred = pipeline.predict(X_test)
accuracy_score(y_test,y_pred)

0.9285714285714286

In [44]:
confusion_matrix(y_test,y_pred)

array([[48,  2],
       [ 4, 30]])

In [45]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.92      0.96      0.94        50
           1       0.94      0.88      0.91        34

    accuracy                           0.93        84
   macro avg       0.93      0.92      0.93        84
weighted avg       0.93      0.93      0.93        84



In [47]:
# Display the pipeline score
score = pipeline.score(X_test, y_test)
print(f"Accuracy: {score:.4f}")

Accuracy: 0.9286


In [49]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ]
)

In [51]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GaussianNB())
])

In [52]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [53]:
y_pred = pipeline.predict(X_test)
accuracy_score(y_test,y_pred)

1.0

In [55]:
from sklearn.tree import DecisionTreeClassifier

In [56]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

In [57]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [58]:
y_pred = pipeline.predict(X_test)
accuracy_score(y_test,y_pred)

1.0

In [59]:
from sklearn.svm import SVC

In [60]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(kernel = 'linear'))
])

In [61]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [62]:
y_pred = pipeline.predict(X_test)
accuracy_score(y_test,y_pred)

1.0